In [ ]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm
import plotly.graph_objects as go


n_bins = 45

def exp( x , N, A , tau ):
    return N*expon.cdf(x , A , tau) 

def gauss( x , N , mu , sigma):
    return N*norm.cdf(x , mu , sigma)

def exp_gauss_cdf(N_exp , A , tau):
    def fixed_exp( x , N , mu , sigma):
        return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
    return fixed_exp
    
def gauss_exp_cdf(N_g , mu , sigma):
    def fixed_gauss( x , N , A , tau):
        return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
    return fixed_gauss

def exp_fit(cdf, N, A , tau, count , edges):
    
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , A = A , tau = tau )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n





In [ ]:
data = np.genfromtxt("Data/timestamp/2026-01-08.csv", delimiter=',')
print(len(data))

data = data[(data > 2e-6)]

print(len(data))

count, edges = np.histogram(data, bins=n_bins , density=False)

fig = go.Figure()

fig.add_trace(
    go.Bar(x=edges[:-1], y=count, name='Histogram', width=np.diff(edges))
)

fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

8636280
411199


In [ ]:
data = np.genfromtxt("Data/timestamp/2026-01-08.csv", delimiter=',')
print(len(data))

data = data[(data > 2e-7)]

print(len(data))

count, edges = np.histogram(data, bins=n_bins , density=False)



8636280
4591592


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.037e+08 (χ²/ndof = 2468563.6)│              Nfcn = 920              │
│ EDM = nan (Goal: 0.0002)         │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N    │    1e5    │    nan    │            │            │         │         │       │
│ 1 │ A    │     1     │    nan    │            │            │         │         │       │
│ 2 │ tau  │   2e-6    │    nan    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────┐
│     │   N   A tau │
├─────┼─────────────┤
│   N │ nan nan nan │
│   A │ nan nan nan │
│ tau │ nan nan nan │
└─────┴─────────────┘

In [ ]:
del_count = np.empty_like(count)

for i in range(len(count)):
    del_count[i] = np.round(float(count[i]) - float(exp( (edges[i+1] + edges[i])/2 , *n.values)))
    # print(count[i],"\t", exp( (edges[i+1] + edges[i])/2 , *n.values) , float(count[i])/float(exp( (edges[i+1] + edges[i])/2 , *n.values)))
    if del_count[i]< 0 : del_count[i] = 0

print(del_count)

m = gauss_fit( gauss , 1e6 , 1.5e-6 , 0.3e-6 , del_count , edges)
# display(m)

[815056 548796 383396 319890 271248 221814 195510 163485 152733 137576
 124629 126311 120979 107715 102454  89182  81857  70746  57964  51552
  45390  38118  35574  30123  28501  25526  22178  21340  19390  17394
  17094  15049  14397  13397  12060  11890  11334  10078   9963   8945
   9092   8570   7939   7742   7615]


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.387e+07 (χ²/ndof = 330199.2)│              Nfcn = 817              │
│ EDM = 0.00285 (Goal: 0.0002)     │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N     │4.941019742e6│0.000000017e6│            │            │         │         │       │
│ 1 │ mu    │645.95446e-9│0.00004e-9 │            │            │         │         │       │
│ 2 │ sigma │332.128863e-9│0.000006e-9│            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────────────────────────────┐
│       │             N            mu         sigma │
├───────┼───────────────────────────────────────────┤
│     N │      0.000301 -498.7574e-24 72.079451e-24 │
│    mu │ -498.7574e-24      1.56e-27    -0.225e-27 │
│ sigma │ 72.079451e-24    -0.225e-27      3.25e-29 │
└───────┴───────────────────────────────────────────┘

In [ ]:
eg_cdf = exp_gauss_cdf( *n.values)

m = gauss_fit( eg_cdf , 1e6 , 1.5e-6 , 0.3e-6 , count , edges)
# display(m)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.387e+07 (χ²/ndof = 330199.2)│              Nfcn = 817              │
│ EDM = 0.00285 (Goal: 0.0002)     │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N     │4.941019742e6│0.000000017e6│            │            │         │         │       │
│ 1 │ mu    │645.95446e-9│0.00004e-9 │            │            │         │         │       │
│ 2 │ sigma │332.128863e-9│0.000006e-9│            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────────────────────────────┐
│       │             N            mu         sigma │
├───────┼───────────────────────────────────────────┤
│     N │      0.000301 -498.7574e-24 72.079451e-24 │
│    mu │ -498.7574e-24      1.56e-27    -0.225e-27 │
│ sigma │ 72.079451e-24    -0.225e-27      3.25e-29 │
└───────┴───────────────────────────────────────────┘

In [ ]:
ge_cdf = gauss_exp_cdf( *m.values)

k = exp_fit( ge_cdf , 7.56e6 , 1e-6 , 2.3e-6 , count , edges)
# display(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2.026e+06 (χ²/ndof = 48238.8)│              Nfcn = 184              │
│ EDM = 1.01e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N    │ 1.0112e6  │ 0.0011e6  │            │            │         │         │       │
│ 1 │ A    │1.28654e-6 │0.00035e-6 │            │            │         │         │       │
│ 2 │ tau  │ 989.8e-9  │  1.6e-9   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬──────────────────────────────────────────────────────────┐
│     │                  N                  A                tau │
├─────┼──────────────────────────────────────────────────────────┤
│   N │           1.28e+06 -86.23823907802e-9  317.5816118829e-9 │
│   A │ -86.23823907802e-9            1.2e-19          -0.09e-18 │
│ tau │  317.5816118829e-9          -0.09e-18           2.55e-18 │
└─────┴──────────────────────────────────────────────────────────┘

In [ ]:

data = np.genfromtxt("Data/timestamp/2026-01-08.csv", delimiter=',')
print(len(data))

data = data[(data > 4.5e-7)]

print(len(data))

count, edges = np.histogram(data, bins= n_bins, density=False)

for _ in range(1000):
    ge_cdf = gauss_exp_cdf( *m.values)
    k = exp_fit( ge_cdf , *k.values , count , edges)

    eg_cdf = exp_gauss_cdf( *k.values)
    m = gauss_fit( eg_cdf , *m.values, count , edges)

display(m)
display(k)

8636280
2878060


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.66e+04 (χ²/ndof = 395.3) │              Nfcn = 50               │
│ EDM = 1.29e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N     │  89.0e6   │   1.5e6   │            │            │         │         │       │
│ 1 │ mu    │ -2.433e-6 │ 0.018e-6  │            │            │         │         │       │
│ 2 │ sigma │ 1.495e-6  │ 0.004e-6  │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────────────────────────────────────────────────────┐
│       │                     N                    mu                 sigma │
├───────┼───────────────────────────────────────────────────────────────────┤
│     N │              2.41e+12 -28.50363119388449e-3  5.730808284897244e-3 │
│    mu │ -28.50363119388449e-3              3.43e-16            -0.071e-15 │
│ sigma │  5.730808284897244e-3            -0.071e-15              1.51e-17 │
└───────┴───────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1.66e+04 (χ²/ndof = 395.3) │              Nfcn = 56               │
│ EDM = 7.94e-10 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N    │  637.6e3  │   1.5e3   │            │            │         │         │       │
│ 1 │ A    │ 1.270e-6  │ 0.001e-6  │            │            │         │         │       │
│ 2 │ tau  │  1.92e-6  │  0.01e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬──────────────────────────────────────────────────────────┐
│     │                  N                  A                tau │
├─────┼──────────────────────────────────────────────────────────┤
│   N │           2.26e+06 -388.3148408916e-9   8.31403767589e-6 │
│   A │ -388.3148408916e-9           1.07e-18           -2.5e-18 │
│ tau │   8.31403767589e-6           -2.5e-18           9.21e-17 │
└─────┴──────────────────────────────────────────────────────────┘